In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv("/content/Metro_Interstate_Traffic_Volume.csv")
print("Dataset Shape:")
print(df.shape)
df.head()

Dataset Shape:
(48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [3]:
print("Columns:")
print(df.columns.tolist())

Columns:
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']


In [4]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
holiday                48143
temp                       0
rain_1h                    0
snow_1h                    0
clouds_all                 0
weather_main               0
weather_description        0
date_time                  0
traffic_volume             0
dtype: int64


In [5]:
df["date_time"] = pd.to_datetime(df["date_time"])
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [6]:
df = df.sort_values("date_time").reset_index(drop=True)
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [7]:
df = df[["date_time", "traffic_volume"]]
df.head()

,date_time,traffic_volume
0,2012-10-02 09:00:00,5545
1,2012-10-02 10:00:00,4516
2,2012-10-02 11:00:00,4767
3,2012-10-02 12:00:00,5026
4,2012-10-02 13:00:00,4918


In [8]:
print(df["traffic_volume"].describe())

count    48204.000000
mean      3259.818355
std       1986.860670
min          0.000000
25%       1193.000000
50%       3380.000000
75%       4933.000000
max       7280.000000
Name: traffic_volume, dtype: float64


In [9]:
df = df.set_index("date_time")
df.head()

,traffic_volume
date_time,
2012-10-02 09:00:00,5545
2012-10-02 10:00:00,4516
2012-10-02 11:00:00,4767
2012-10-02 12:00:00,5026
2012-10-02 13:00:00,4918


In [10]:
data = df["traffic_volume"].values.reshape(-1, 1)
print("Data Shape:")
print(data.shape)

Data Shape:
(48204, 1)


In [11]:
train_size = int(len(data) * 0.80)
train_data = data[:train_size]
test_data = data[train_size:]
print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

Training samples: 38563
Testing samples: 9641


In [12]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [13]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [14]:
SEQUENCE_LENGTH = 24
def create_sequences(data, sequence_length):
    X = []
    y = []
    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length : i])
        y.append(data[i])
    return (np.array(X), np.array(y))

In [15]:
X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (38539, 24, 1)
y_train shape: (38539, 1)


In [16]:
combined_data = np.concatenate((train_scaled[-SEQUENCE_LENGTH:], test_scaled), axis=0)
X_test, y_test = create_sequences(combined_data, SEQUENCE_LENGTH)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (9641, 24, 1)
y_test shape: (9641, 1)


In [17]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
        LSTM(32),
        Dense(1),
    ]
)
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(optimizer="adam", loss="mean_squared_error")

In [19]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [20]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 0.0180 - val_loss: 0.0061
Epoch 2/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0090 - val_loss: 0.0065
Epoch 3/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.0082 - val_loss: 0.0058
Epoch 4/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0073 - val_loss: 0.0045
Epoch 5/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.0068 - val_loss: 0.0050
Epoch 6/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0065 - val_loss: 0.0039
Epoch 7/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0063 - val_loss: 0.0040
Epoch 8/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.0061 - val_loss: 0.0041
Epoch 9/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0061 - val_loss: 0.0038
Epoch 10/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0060 - val_loss: 0.0043
Epoch 11/30
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0058 - val_loss: 0.0038
Epoch 12/30
1084/1084 ━━━━━

In [21]:
test_loss = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", test_loss)

302/302 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0029
Test Loss: 0.0029022558592259884


In [22]:
predictions_scaled = model.predict(X_test)
print("Prediction shape:", predictions_scaled.shape)

302/302 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (9641, 1)


In [23]:
predictions = scaler.inverse_transform(predictions_scaled)
actual_values = scaler.inverse_transform(y_test)

In [24]:
mae = mean_absolute_error(actual_values, predictions)
print(f"MAE: {mae:.2f} vehicles/hour")

MAE: 265.47 vehicles/hour


In [25]:
mse = mean_squared_error(actual_values, predictions)
print(f"MSE: {mse:.4f}")

MSE: 153814.8944


In [26]:
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.4f} kW")

RMSE: 392.1924 kW


In [27]:
results = pd.DataFrame(
    {
        "Actual Power (kW)": actual_values.flatten(),
        "Predicted Power (kW)": predictions.flatten(),
    }
)
results["Absolute Error (kW)"] = abs(
    results["Actual Power (kW)"] - results["Predicted Power (kW)"]
)
results.head(20)

,Actual Power (kW),Predicted Power (kW),Absolute Error (kW)
0,2704.0,2599.161133,104.838867
1,2704.0,2447.772217,256.227783
2,2204.0,2524.458984,320.458984
3,2204.0,1937.361084,266.638916
4,1713.0,1868.899536,155.899536
5,1713.0,1470.384399,242.615601
6,1068.0,1401.396484,333.396484
7,611.0,822.251282,211.251282
8,362.0,355.477509,6.522491
9,254.0,220.378586,33.621414


In [28]:
model.save("model.keras")